# EC2 & Compute

## EC2 Instance Types

EC2 instances come in different families optimized for different workloads:

**General Purpose (t3, m5)** balance compute, memory, and networking. Suitable for web servers, small databases, and development environments.

**Compute Optimized (c5, c6)** have high CPU performance. Use for batch processing, media transcoding, and high-performance web applications.

**Memory Optimized (r5, x1)** have large amounts of RAM. Use for in-memory databases, caches, and real-time analytics.

**Storage Optimized (i3, h1)** have high sequential read/write access to large data sets. Use for NoSQL databases and data warehousing.

**Accelerated Computing (p3, g4)** include GPUs or FPGAs. Use for machine learning, graphics rendering, and scientific computing.

## Amazon Machine Images (AMIs)

An **AMI** is a pre-configured template containing the OS, applications, and configurations. AWS provides public AMIs (Amazon Linux, Ubuntu, Windows), or you can create custom AMIs from running instances.

## Security Groups and Key Pairs

**Security Groups** act as virtual firewalls controlling inbound and outbound traffic. They operate at the instance level and are stateful (if you allow inbound traffic, outbound response is automatically allowed).

**Key Pairs** are used for SSH access to Linux instances or RDP to Windows instances. The private key is downloaded once and must be kept secure.

## Hands-On: Launch and Connect to EC2

Create a key pair:

```bash
aws ec2 create-key-pair --key-name my-key --query 'KeyMaterial' \
  --output text > my-key.pem
chmod 400 my-key.pem
```

Create a security group:

```bash
aws ec2 create-security-group --group-name web-sg \
  --description "Security group for web servers"
```

Allow SSH access:

```bash
aws ec2 authorize-security-group-ingress --group-name web-sg \
  --protocol tcp --port 22 --cidr 0.0.0.0/0
```

Allow HTTP and HTTPS:

```bash
aws ec2 authorize-security-group-ingress --group-name web-sg \
  --protocol tcp --port 80 --cidr 0.0.0.0/0

aws ec2 authorize-security-group-ingress --group-name web-sg \
  --protocol tcp --port 443 --cidr 0.0.0.0/0
```

Launch an EC2 instance:

```bash
aws ec2 run-instances --image-id ami-0c55b159cbfafe1f0 \
  --instance-type t3.micro --key-name my-key \
  --security-groups web-sg --count 1
```

## User Data Script

User data scripts run when an instance launches. Use them to install software and configure the instance:

```bash
aws ec2 run-instances --image-id ami-0c55b159cbfafe1f0 \
  --instance-type t3.micro --key-name my-key \
  --user-data file://init-script.sh
```

Example init-script.sh:

```bash
#!/bin/bash
yum update -y
yum install -y httpd
systemctl start httpd
echo "<h1>Hello from $(hostname -f)</h1>" > /var/www/html/index.html
```

## Python Boto3 Example

In [ ]:
import boto3

ec2 = boto3.client('ec2')

# Launch instance
response = ec2.run_instances(
    ImageId='ami-0c55b159cbfafe1f0',
    MinCount=1,
    MaxCount=1,
    InstanceType='t3.micro',
    KeyName='my-key',
    SecurityGroups=['web-sg']
)

instance_id = response['Instances'][0]['InstanceId']
print(f"Launched instance: {instance_id}")

# Describe instances
instances = ec2.describe_instances(InstanceIds=[instance_id])
for reservation in instances['Reservations']:
    for instance in reservation['Instances']:
        print(f"State: {instance['State']['Name']}")
        print(f"Public IP: {instance.get('PublicIpAddress', 'N/A')}")

## Terraform Example

```hcl
resource "aws_instance" "web" {
  ami           = "ami-0c55b159cbfafe1f0"
  instance_type = "t3.micro"
  key_name      = aws_key_pair.deployer.key_name

  vpc_security_group_ids = [aws_security_group.web.id]

  user_data = <<-EOF
              #!/bin/bash
              yum update -y
              yum install -y httpd
              systemctl start httpd
              EOF

  tags = {
    Name = "web-server"
  }
}

resource "aws_security_group" "web" {
  name = "web-sg"

  ingress {
    from_port   = 80
    to_port     = 80
    protocol    = "tcp"
    cidr_blocks = ["0.0.0.0/0"]
  }

  ingress {
    from_port   = 22
    to_port     = 22
    protocol    = "tcp"
    cidr_blocks = ["0.0.0.0/0"]
  }
}
```

## Quiz 1

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ Which EC2 instance family is best for web servers?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q3847291" value="0">
      <span>Compute Optimized</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q3847291" value="1">
      <span>General Purpose</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q3847291" value="2">
      <span>Memory Optimized</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q3847291" value="3">
      <span>Storage Optimized</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 2

<div class="quiz" data-correct="0">
  <p class="font-semibold mb-3">❓ What is an AMI?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7293847" value="0">
      <span>A pre-configured template with OS and applications</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7293847" value="1">
      <span>A security group</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7293847" value="2">
      <span>A network interface</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7293847" value="3">
      <span>A storage volume</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 3

<div class="quiz" data-correct="2">
  <p class="font-semibold mb-3">❓ What is the purpose of a Security Group?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4829374" value="0">
      <span>To encrypt data</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4829374" value="1">
      <span>To manage user access</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4829374" value="2">
      <span>To control inbound and outbound traffic</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4829374" value="3">
      <span>To monitor instance performance</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 4

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What is user data in EC2?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5738291" value="0">
      <span>Information about the instance owner</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5738291" value="1">
      <span>A script that runs when the instance launches</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5738291" value="2">
      <span>The instance's IP address</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5738291" value="3">
      <span>The instance's storage configuration</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 5

<div class="quiz" data-correct="0">
  <p class="font-semibold mb-3">❓ What is a key pair used for?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374629" value="0">
      <span>SSH access to Linux instances</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374629" value="1">
      <span>Encrypting data at rest</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374629" value="2">
      <span>Managing IAM permissions</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374629" value="3">
      <span>Configuring security groups</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>